In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────────
import pickle
import warnings
from pathlib import Path
from time import perf_counter as pc

# ─── Scientific Computing ───────────────────────────────────────────────────────
import math
import numpy as np
import fast_histogram
import pandas as pd

# ─── Geospatial Processing ──────────────────────────────────────────────────────
import geopandas as gpd
import pyproj
from scipy.spatial import cKDTree
from shapely.geometry import LineString, MultiLineString, Point

# ─── Geocoding (Geopy) ──────────────────────────────────────────────────────────
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ─── Network Analysis ───────────────────────────────────────────────────────────
import pandana

# ─── Mapping and Visualization ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ─── Custom Modules ────────────────────────────────────────────────────────────
import geo_util as gu

In [ ]:
# global definitions
_data_path = Path( './data/' )

In [ ]:
net = pandana.Network.from_hdf5(_data_path.joinpath('nl_pandana_network.h5'))

In [ ]:
pop = gpd.read_parquet(_data_path.joinpath('HDX_NL.parquet')).to_crs(gu.projectedCrsNL)

In [ ]:
pop = gu.assignNearestPandanaNodesWithGeometry( pop, net, geometry_column='geometry', columnPrefix='nearest_node' )

In [ ]:
gu.plotFastHistogram(pop['nearest_node_distance'])

In [ ]:
def plotWeightedHistogram(
    values: np.ndarray | pd.Series,
    weights: np.ndarray | pd.Series,
    bins: int = 20,
    ax: plt.Axes = None,
    title: str = 'Weighted histogram',
    xlabel: str = 'Value',
    ylabel: str = 'Weighted count',
    color: str = 'tab:blue'
) -> plt.Axes:
    """
    Plots a weighted histogram using fast_histogram.

    Args:
        values: Array of values to bin (e.g., distances).
        weights: Array of weights to sum in each bin (e.g., headcounts).
        bins: Number of histogram bins.
        ax: Matplotlib Axes to plot on. If None, creates a new one.
        title: Plot title.
        xlabel: Label for x-axis.
        ylabel: Label for y-axis.
        color: Color of bars.

    Returns:
        The matplotlib Axes with the plotted histogram.
    """
    values = np.asarray(values)
    weights = np.asarray(weights)
    vmin, vmax = values.min(), values.max()

    counts = fast_histogram.histogram1d(
        values,
        bins=bins,
        range=(vmin, vmax),
        weights=weights
    )

    edges = np.linspace(vmin, vmax, bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    if ax is None:
        _, ax = plt.subplots()

    ax.bar(centers, counts, width=(vmax - vmin) / bins, edgecolor='black', color=color)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    return ax


In [ ]:
plotWeightedHistogram(
    pop.nearest_node_distance, 
    pop.Population, 
    title='Pupulation per distance',
    xlabel='distance in meters',
    ylabel='headcount'
)

In [ ]:
plotWeightedHistogram(
    pop[pop.nearest_node_distance>500].nearest_node_distance, 
    pop[pop.nearest_node_distance>500].Population, 
    title='Pupulation per distance',
    xlabel='distance in meters',
    ylabel='headcount'
)

In [ ]:
plotWeightedHistogram(
    pop[pop.nearest_node_distance>4000].nearest_node_distance, 
    pop[pop.nearest_node_distance>4000].Population, 
    title='Pupulation per distance',
    xlabel='distance in meters',
    ylabel='headcount'
)

In [ ]:
pop.nlargest(20, 'nearest_node_distance').explore(style_kwds={'color': 'red', 'weight': 8})

In [ ]:
furthest_pop_index = pop['nearest_node_distance'].idxmax()
pop.loc[[furthest_pop_index]].explore( style_kwds={'color': 'red', 'weight': 8} )

In [ ]:
group_pop = pop[pop.nearest_node_id == pop.nearest_node_id.value_counts().idxmax()].index
pop.loc[group_pop].explore()

In [ ]:
dan_banks = pd.read_parquet(_data_path.joinpath('merged_banks.parquet'))
dan_banks = gpd.GeoDataFrame(
    dan_banks,
    geometry=gpd.points_from_xy(dan_banks['Longitude'], dan_banks['Latitude']),
    crs='EPSG:4326'  # assuming lat/lon in WGS84
)

In [ ]:
assert np.allclose(
    dan_banks[['Longitude', 'Latitude']].dropna().values,
    np.column_stack((dan_banks.geometry.x, dan_banks.geometry.y)), 
    rtol=1e-8, atol=1e-10
) 

In [ ]:
banks = dan_banks.to_crs(gu.projectedCrsNL)

In [ ]:
banks = gu.assignNearestPandanaNodesWithGeometry( banks, net, geometry_column='geometry', columnPrefix='nearest_node' )

In [ ]:
gu.plotFastHistogram(banks['nearest_node_distance'])

In [ ]:
banks.nearest_node_distance.describe()

In [ ]:
furthest_index = banks['nearest_node_distance'].idxmax()
furthest_row = banks.loc[furthest_index]

In [ ]:
furthest_address = furthest_row.FullAddress.replace('\xa0', ' ')
furthest_address

In [ ]:
# Initialize geocoder
geolocator = Nominatim(user_agent='thesis Dan')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

located = geocode( furthest_address )
xy = (located.longitude, located.latitude)

In [ ]:
m, dist = gu.compareGeocodedAndNetworkLocations(
    banks=banks,
    furthest_index=furthest_index,
    geocoded_xy=xy  # (lon, lat)
)
m

In [ ]:
banks = gu.geocodeTopNIntoGeometryColumn(
    banks,
    address_column='FullAddress',
    sort_column='nearest_node_distance',
    new_geometry_column='geocoded_geometry',
    top_n=3
)

In [ ]:
banks[banks['geocoded_geometry'] != banks['geometry']]

In [ ]:
# Filter mismatches
mismatched = banks[banks['geocoded_geometry'] != banks['geometry']].copy()

# Create GeoDataFrame for original and geocoded points
original = mismatched.copy()
original.set_geometry('geometry', inplace=True)

geocoded = mismatched.copy()
geocoded.set_geometry('geocoded_geometry', inplace=True)

# Base map with original locations
map_view = original.explore(color='red', marker_kwds={'radius': 6}, name='Original Geometry')

# Add geocoded points
geocoded.explore(m=map_view, color='green', marker_kwds={'radius': 6}, name='Geocoded Geometry')

# Create LineStrings connecting original and geocoded points
lines = mismatched.apply(
    lambda row: LineString([row['geometry'], row['geocoded_geometry']]), axis=1
)
line_gdf = gpd.GeoDataFrame(
    mismatched[['FullAddress']],  # retain address for tooltip
    geometry=lines,
    crs=banks.crs
)

# Add lines with tooltip showing FullAddress
line_gdf.explore(
    m=map_view,
    color='blue',
    name='Shift Line',
    tooltip='FullAddress'
)

# Optional: add layer control (in Jupyter)
import folium
folium.LayerControl().add_to(map_view)

map_view


In [ ]:
def aggregatePopulationToCentroids(population: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Aggregates population data by 'nearest_node_id', computing attribute sums and centroid geometries.

    Args:
        population: GeoDataFrame with Point geometries and a 'nearest_node_id' column.

    Returns:
        GeoDataFrame with one row per 'nearest_node_id', summed attributes, and centroid geometry.
    """
    # Numeric aggregation
    agg_dict = {
        'Population': 'sum',
        'Children Under Five': 'sum',
        'Elderly 60 Plus': 'sum',
        'Men': 'sum',
        'Women': 'sum',
        'Women of Reproductive Age 15 49': 'sum',
        'Youth 15 24': 'sum',
        'nearest_node_distance': 'mean',
    }

    # Coordinate means (centroid proxy)
    coords = population[['nearest_node_id', 'geometry']].copy()
    coords['x'] = coords.geometry.x
    coords['y'] = coords.geometry.y
    centroid_coords = coords.groupby('nearest_node_id')[['x', 'y']].mean()

    # Rebuild as geometry
    centroid_geom = gpd.GeoSeries.from_xy(
        centroid_coords['x'],
        centroid_coords['y'],
        crs=population.crs
    )

    # Combine with attribute aggregation
    attributes = (
        population
        .groupby('nearest_node_id')
        .agg(agg_dict)
        .assign(num_points=population.groupby('nearest_node_id').size())
    )

    # Join
    pop_agg_centroids = gpd.GeoDataFrame(attributes, geometry=centroid_geom)
    pop_agg_centroids = pop_agg_centroids.reset_index()

    return pop_agg_centroids

In [ ]:
def aggregateBanksToCentroids(banks: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Aggregates bank data by 'nearest_node_id', computing representative attributes, mean distance,
    and centroid geometry.

    Args:
        banks: GeoDataFrame with Point geometries and a 'nearest_node_id' column.

    Returns:
        GeoDataFrame with one row per 'nearest_node_id', including:
        - representative address info (first entry),
        - number of banks,
        - average distance to network,
        - centroid geometry.
    """
    # Centroid of grouped geometries
    coords = banks[['nearest_node_id', 'geometry']].copy()
    coords['x'] = coords.geometry.x
    coords['y'] = coords.geometry.y
    centroid_coords = coords.groupby('nearest_node_id')[['x', 'y']].mean()

    centroid_geom = gpd.GeoSeries.from_xy(
        centroid_coords['x'],
        centroid_coords['y'],
        crs=banks.crs
    )

    # Aggregation logic
    agg_dict = {
        'Bank': 'first',
        'Address': 'first',
        'PostalCode': 'first',
        'City': 'first',
        'FullAddress': 'first',
        'nearest_node_distance': 'mean',
    }

    attributes = (
        banks
        .groupby('nearest_node_id')
        .agg(agg_dict)
        .assign(num_banks=banks.groupby('nearest_node_id').size())
    )

    # Combine with geometry
    bank_agg_centroids = gpd.GeoDataFrame(attributes, geometry=centroid_geom)
    bank_agg_centroids = bank_agg_centroids.reset_index()

    return bank_agg_centroids

In [ ]:
pop_agg = aggregatePopulationToCentroids(pop)

In [ ]:
pop_agg.shape,pop.shape

In [ ]:
banks_agg = aggregateBanksToCentroids(banks)

In [ ]:
banks_agg.shape, banks.shape

In [ ]:
use_pop = pop_agg.reset_index().rename(columns={'index': 'pop_idx'})
use_banks = banks_agg.reset_index().rename(columns={'index': 'poi_idx'})

In [ ]:
use_pop.to_parquet(_data_path.joinpath('population_aggregated_per_node.parquet'))
use_banks.to_parquet(_data_path.joinpath('banks_aggregated_per_node.parquet'))

In [ ]:
max_dist_threshold = 20_000  # in meters
dist = gu.computeAccessMatrix(use_pop, use_banks, net, max_dist_threshold, batch_size=100_000_000, num_fallback_poi=10)

In [ ]:
f'{dist.shape[0]:,}'

In [ ]:
dist.to_parquet(_data_path.joinpath(f'access_matrix_{max_dist_threshold}.parquet'))

In [ ]:
n_pop = dist['pop_idx'].nunique()
n_poi = dist['poi_idx'].nunique()
num_cells = n_pop * n_poi

# Estimate memory: float64 = 8 bytes per value
memory_bytes = num_cells * 8
memory_mb = memory_bytes / 2**20
memory_gb = memory_bytes / 2**30

print(f'Dense matrix shape would be: {n_pop:,} x {n_poi:,} = {num_cells:,} cells')
print(f'Estimated memory: {memory_mb:,.1f} MB ({memory_gb:,.2f} GB)')

In [ ]:
assert not dist.duplicated(subset=['pop_idx', 'poi_idx']).any(), 'Duplicate pop-poi pairs found!'

In [ ]:
dist_matrix = gu.SparseDistanceMatrix(
    dist,
    pop_col='pop_idx',
    poi_col='poi_idx',
    dist_col='total_dist'
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes = axes.T  # So column-wise: axes[0] = left column, axes[1] = right column

columns = {
    'left': ['road_distance', 'total_dist'],
    'right': ['pop_to_node_dist', 'poi_to_node_dist']
}

for col_idx, (col_group, col_names) in enumerate(columns.items()):
    for row_idx, col_name in enumerate(col_names):
        gu.plotFastHistogram(
            dist[col_name].dropna().to_numpy(),
            ax=axes[col_idx][row_idx],
            title=col_name
        )

plt.tight_layout()
plt.show()


In [ ]:
m = gu.visualizePopPoiConnection(
    row_idx=dist['total_dist'].idxmax(),
    all_distances=dist,
    population=use_pop,
    points_of_interest=use_banks,
    network=net,
    pop_id_col='pop_idx',
    poi_id_col='poi_idx'
)
m